# Version 6.4 : LightGBM Optimized (Feature Selection + Hyperparameter Tuning)

**Amélioration de V6.3** :
1. 📊 Feature selection basée sur l'importance
2. 🎯 Hyperparameter tuning avec Optuna (100 trials)
3. 🚀 Re-training avec les meilleurs paramètres

**Objectif** : C-index > 0.74 (améliorer V6.3)

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import optuna

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported")

✓ Libraries imported


## 2. Data Loading & Feature Engineering

In [2]:
DATA_PATH = r"C:\Users\guill\Desktop\Data Challenge QRT\Data-Challenge-Prediction-de-Survie"

# Load data
clinical_train = pd.read_csv(f"{DATA_PATH}\\X_train\\clinical_train.csv")
target_train = pd.read_csv(f"{DATA_PATH}\\target_train.csv")
molecular_train = pd.read_csv(f"{DATA_PATH}\\X_train\\molecular_train.csv")

print("✓ Data loaded")

✓ Data loaded


In [3]:
# Feature engineering (same as V6.3)
def create_cytogenetic_features(clinical_df):
    cyto_features = pd.DataFrame(index=clinical_df['ID'])
    cyto_col = clinical_df.set_index('ID')['CYTOGENETICS'].fillna('')
    cyto_features['cyto_del_count'] = cyto_col.str.count(r'del\(')
    cyto_features['cyto_has_del'] = (cyto_features['cyto_del_count'] > 0).astype(int)
    cyto_features['cyto_transloc_count'] = cyto_col.str.count(r't\(')
    cyto_features['cyto_has_transloc'] = (cyto_features['cyto_transloc_count'] > 0).astype(int)
    cyto_features['cyto_inv_count'] = cyto_col.str.count(r'inv\(')
    cyto_features['cyto_has_inv'] = (cyto_features['cyto_inv_count'] > 0).astype(int)
    cyto_features['cyto_gain_count'] = cyto_col.str.count(r'\+')
    cyto_features['cyto_has_gain'] = (cyto_features['cyto_gain_count'] > 0).astype(int)
    cyto_features['cyto_loss_count'] = cyto_col.str.count(r'-[0-9XY]')
    cyto_features['cyto_has_loss'] = (cyto_features['cyto_loss_count'] > 0).astype(int)
    cyto_features['cyto_other_count'] = cyto_col.str.count(r'add\(|ins\(|dup\(')
    cyto_features['cyto_total_anomalies'] = (
        cyto_features['cyto_del_count'] + cyto_features['cyto_transloc_count'] + 
        cyto_features['cyto_inv_count'] + cyto_features['cyto_gain_count'] + 
        cyto_features['cyto_loss_count'] + cyto_features['cyto_other_count']
    )
    cyto_features['cyto_normal'] = cyto_col.str.match(r'^46,(xx|xy)(\[\d+\])?$', case=False).astype(int)
    cyto_features['cyto_complex'] = (
        (cyto_features['cyto_total_anomalies'] >= 3) | 
        cyto_col.str.contains('complex', case=False, na=False)
    ).astype(int)
    chromosomes = [str(i) for i in range(1, 23)] + ['X', 'Y']
    for chrom in chromosomes:
        pattern = rf'(\b|[,\(]){chrom}([,;:\)\[]|[pq])'
        cyto_features[f'cyto_chr{chrom}_affected'] = cyto_col.str.contains(
            pattern, case=False, na=False, regex=True
        ).astype(int)
    cyto_features['cyto_monosomy7'] = cyto_col.str.contains(r'-7[^0-9]|^45.*-7', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_trisomy8'] = cyto_col.str.contains(r'\+8[^0-9]|^47.*\+8', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del5q'] = cyto_col.str.contains(r'del\(5\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del20q'] = cyto_col.str.contains(r'del\(20\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr3_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(3[;,:\)]', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr7_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(7[;,:\)]', case=False, na=False, regex=True).astype(int)
    return cyto_features.fillna(0)

def create_molecular_features(molecular_df, patient_ids, top_n_genes=20):
    mol_features = pd.DataFrame({'ID': patient_ids})
    mutation_counts = molecular_df.groupby('ID').size().to_frame('mutation_count_total')
    mol_features = mol_features.merge(mutation_counts, on='ID', how='left')
    vaf_stats = molecular_df.groupby('ID')['VAF'].agg([
        ('vaf_mean', 'mean'), ('vaf_max', 'max'), ('vaf_sum', 'sum')
    ]).reset_index()
    mol_features = mol_features.merge(vaf_stats, on='ID', how='left')
    effect_counts = molecular_df.groupby(['ID', 'EFFECT']).size().unstack(fill_value=0)
    effect_counts.columns = [f'effect_{col}' for col in effect_counts.columns]
    mol_features = mol_features.merge(effect_counts.reset_index(), on='ID', how='left')
    top_genes_list = molecular_df['GENE'].value_counts().head(top_n_genes).index.tolist()
    for gene in top_genes_list:
        gene_mutations = molecular_df[molecular_df['GENE'] == gene].groupby('ID').size()
        mol_features[f'gene_{gene}_count'] = mol_features['ID'].map(gene_mutations)
        gene_present = molecular_df[molecular_df['GENE'] == gene]['ID'].unique()
        mol_features[f'gene_{gene}_present'] = mol_features['ID'].isin(gene_present).astype(int)
    feature_cols = [col for col in mol_features.columns if col != 'ID']
    mol_features[feature_cols] = mol_features[feature_cols].fillna(0)
    return mol_features.set_index('ID')

def create_advanced_features(X_df):
    X_adv = X_df.copy()
    X_adv['blast_to_wbc'] = X_adv['BM_BLAST'] / (X_adv['WBC'] + 1)
    X_adv['monocyte_ratio'] = X_adv['MONOCYTES'] / (X_adv['WBC'] + 1)
    X_adv['anc_ratio'] = X_adv['ANC'] / (X_adv['WBC'] + 1)
    X_adv['platelet_to_blast'] = X_adv['PLT'] / (X_adv['BM_BLAST'] + 1)
    X_adv['hb_to_plt'] = X_adv['HB'] / (X_adv['PLT'] + 1)
    X_adv['vaf_mutation_burden'] = X_adv['vaf_mean'] * X_adv['mutation_count_total']
    X_adv['vaf_per_mutation'] = X_adv['vaf_sum'] / (X_adv['mutation_count_total'] + 1)
    X_adv['cytogenetic_risk_score'] = (
        X_adv['cyto_complex'] * 3 + X_adv['cyto_chr7_affected'] * 2 + 
        X_adv['cyto_del5q'] * 1.5 + X_adv['cyto_loss_count'] * 0.5
    )
    X_adv['blast_cytogenetic_risk'] = X_adv['BM_BLAST'] * X_adv['cytogenetic_risk_score']
    X_adv['blast_to_mutation'] = X_adv['BM_BLAST'] / (X_adv['mutation_count_total'] + 1)
    X_adv['wbc_plt_index'] = X_adv['WBC'] * X_adv['PLT'] / 1000
    X_adv['blast_hb_ratio'] = X_adv['BM_BLAST'] / (X_adv['HB'] + 1)
    X_adv['monocyte_blast_ratio'] = X_adv['MONOCYTES'] / (X_adv['BM_BLAST'] + 1)
    X_adv['mutation_per_vaf'] = X_adv['mutation_count_total'] / (X_adv['vaf_mean'] + 0.01)
    X_adv['cyto_anomaly_density'] = X_adv['cyto_total_anomalies'] / (X_adv['cyto_total_anomalies'].max() + 1)
    X_adv['blast_mutation_interaction'] = X_adv['BM_BLAST'] * X_adv['mutation_count_total']
    X_adv['blast_vaf_interaction'] = X_adv['BM_BLAST'] * X_adv['vaf_mean']
    X_adv['blast_cyto_complex'] = X_adv['BM_BLAST'] * X_adv['cyto_complex']
    X_adv['tp53_blast'] = X_adv['gene_TP53_present'] * X_adv['BM_BLAST']
    X_adv['runx1_mutation_burden'] = X_adv['gene_RUNX1_count'] * X_adv['mutation_count_total']
    X_adv['nras_vaf'] = X_adv['gene_NRAS_present'] * X_adv['vaf_mean']
    X_adv['cyto_mutation_interaction'] = X_adv['cyto_total_anomalies'] * X_adv['mutation_count_total']
    X_adv['chr7_blast'] = X_adv['cyto_chr7_affected'] * X_adv['BM_BLAST']
    X_adv['vaf_cyto_burden'] = X_adv['vaf_sum'] * X_adv['cyto_total_anomalies']
    X_adv['vaf_tp53'] = X_adv['vaf_mean'] * X_adv['gene_TP53_present']
    X_adv['log_wbc'] = np.log1p(X_adv['WBC'])
    X_adv['log_plt'] = np.log1p(X_adv['PLT'])
    X_adv['log_blast'] = np.log1p(X_adv['BM_BLAST'])
    X_adv['log_mutation_count'] = np.log1p(X_adv['mutation_count_total'])
    X_adv['log_vaf_sum'] = np.log1p(X_adv['vaf_sum'])
    return X_adv

# Create features
cyto_features_train = create_cytogenetic_features(clinical_train)
train_patient_ids = clinical_train['ID'].unique()
mol_features_train = create_molecular_features(molecular_train, train_patient_ids)

target_clean = target_train.dropna(subset=['OS_YEARS', 'OS_STATUS']).copy()
target_clean['OS_STATUS'] = target_clean['OS_STATUS'].astype(bool)
target_clean = target_clean.set_index('ID')

clinical_train_clean = clinical_train[clinical_train['ID'].isin(target_clean.index)].copy()
clinical_train_clean = clinical_train_clean.set_index('ID').loc[target_clean.index]

numeric_features = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
X_numeric = clinical_train_clean[numeric_features].copy()
center_encoded = pd.get_dummies(clinical_train_clean['CENTER'], prefix='CENTER', drop_first=True)
X_clinical = pd.concat([X_numeric, center_encoded], axis=1)

mol_features_train_aligned = mol_features_train.reindex(X_clinical.index, fill_value=0)
cyto_features_train_aligned = cyto_features_train.reindex(X_clinical.index, fill_value=0)
X_all_base = pd.concat([X_clinical, mol_features_train_aligned, cyto_features_train_aligned], axis=1)
X_all = create_advanced_features(X_all_base)

y_surv = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_clean)

print(f"✓ Features created: {X_all.shape[1]} total")

✓ Features created: 162 total


## 3. Preprocessing & Initial Split

In [4]:
# Impute missing values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_all),
    index=X_all.index,
    columns=X_all.columns
)

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X_imputed, y_surv, test_size=0.3, random_state=42,
    stratify=target_clean['OS_STATUS'].astype(int)
)

print(f"✓ Split: {len(X_train)} train, {len(X_val)} val")

✓ Split: 2221 train, 952 val


## 4. Baseline Model for Feature Importance

In [5]:
print("="*60)
print("BASELINE MODEL (for feature selection)")
print("="*60)

# Baseline params
baseline_params = {
    'objective': 'regression',
    'metric': 'l2',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1,
    'random_state': 42
}

# Prepare target
y_train_lgbm = np.log1p(y_train['OS_YEARS'])
y_val_lgbm = np.log1p(y_val['OS_YEARS'])

# Train baseline
train_data = lgb.Dataset(X_train, label=y_train_lgbm)
val_data = lgb.Dataset(X_val, label=y_val_lgbm, reference=train_data)

baseline_model = lgb.train(
    baseline_params,
    train_data,
    num_boost_round=300,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

# Get baseline C-index
y_pred_baseline = -baseline_model.predict(X_val, num_iteration=baseline_model.best_iteration)
c_baseline = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_baseline
)[0]

print(f"✓ Baseline C-index: {c_baseline:.4f}")
print(f"  Features: {X_train.shape[1]}")

BASELINE MODEL (for feature selection)
✓ Baseline C-index: 0.6904
  Features: 162


## 5. Feature Selection (Top 90% Cumulative Importance)

In [6]:
print("\n" + "="*60)
print("FEATURE SELECTION")
print("="*60)

# Get feature importance
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': baseline_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False).reset_index(drop=True)

# Calculate cumulative importance
total_importance = importance_df['importance'].sum()
importance_df['cumulative_pct'] = importance_df['importance'].cumsum() / total_importance

# Select top features (90% cumulative importance)
n_selected = (importance_df['cumulative_pct'] <= 0.90).sum()
selected_features = importance_df.head(n_selected)['feature'].tolist()

print(f"\n✓ Selected {n_selected} features (90% importance)")
print(f"  Reduction: {X_train.shape[1]} → {n_selected} features")
print(f"\n  Top 10: {selected_features[:10]}")

# Filter datasets
X_train_sel = X_train[selected_features]
X_val_sel = X_val[selected_features]


FEATURE SELECTION

✓ Selected 34 features (90% importance)
  Reduction: 162 → 34 features

  Top 10: ['HB', 'cyto_mutation_interaction', 'PLT', 'platelet_to_blast', 'CENTER_KI', 'wbc_plt_index', 'blast_mutation_interaction', 'ANC', 'MONOCYTES', 'mutation_per_vaf']


## 6. Hyperparameter Optimization with Optuna

In [7]:
print("\n" + "="*60)
print("HYPERPARAMETER OPTIMIZATION (100 TRIALS)")
print("="*60)

def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'l2',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1,
        'random_state': 42
    }
    
    # Train
    train_data = lgb.Dataset(X_train_sel, label=y_train_lgbm)
    val_data = lgb.Dataset(X_val_sel, label=y_val_lgbm, reference=train_data)
    
    model = lgb.train(
        params,
        train_data,
        num_boost_round=500,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    # Predict and evaluate
    y_pred = -model.predict(X_val_sel, num_iteration=model.best_iteration)
    c_index = concordance_index_censored(
        y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred
    )[0]
    
    return c_index

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = study.best_params
best_c_index = study.best_value

print(f"\n✓ Optimization complete")
print(f"\nBest hyperparameters:")
for param, value in best_params.items():
    print(f"  {param}: {value:.4f}" if isinstance(value, float) else f"  {param}: {value}")
print(f"\nBest C-index: {best_c_index:.4f}")


HYPERPARAMETER OPTIMIZATION (100 TRIALS)


  0%|          | 0/100 [00:00<?, ?it/s]


✓ Optimization complete

Best hyperparameters:
  learning_rate: 0.0613
  num_leaves: 80
  max_depth: 5
  min_child_samples: 12
  subsample: 0.9969
  colsample_bytree: 0.7500
  reg_alpha: 0.2155
  reg_lambda: 0.3008

Best C-index: 0.6959


## 7. Train Final Optimized Model

In [8]:
print("\n" + "="*60)
print("FINAL OPTIMIZED MODEL")
print("="*60)

# Add fixed params
final_params = best_params.copy()
final_params.update({
    'objective': 'regression',
    'metric': 'l2',
    'boosting_type': 'gbdt',
    'verbose': -1,
    'random_state': 42
})

# Train final model
train_data = lgb.Dataset(X_train_sel, label=y_train_lgbm)
val_data = lgb.Dataset(X_val_sel, label=y_val_lgbm, reference=train_data)

final_model = lgb.train(
    final_params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100)
    ]
)

# Final predictions
y_pred_train = -final_model.predict(X_train_sel, num_iteration=final_model.best_iteration)
y_pred_val = -final_model.predict(X_val_sel, num_iteration=final_model.best_iteration)

c_train = concordance_index_censored(
    y_train['OS_STATUS'], y_train['OS_YEARS'], y_pred_train
)[0]

c_val = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_val
)[0]

print(f"\n✓ Training complete")
print(f"  Best iteration: {final_model.best_iteration}")
print(f"  C-index (Train): {c_train:.4f}")
print(f"  C-index (Val):   {c_val:.4f}")


FINAL OPTIMIZED MODEL
[100]	train's l2: 0.202362	val's l2: 0.369248

✓ Training complete
  Best iteration: 57
  C-index (Train): 0.7842
  C-index (Val):   0.6959


## 8. Performance Comparison

In [ ]:
print("\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)

comparison = pd.DataFrame([
    {'Model': 'V4 RSF', 'Features': 90, 'C-index': 0.7404},
    {'Model': 'V6.1 XGBoost', 'Features': 'Auto', 'C-index': 0.7413},
    {'Model': 'V6.3 LightGBM Baseline', 'Features': X_train.shape[1], 'C-index': c_baseline},
    {'Model': 'V6.4 LightGBM Optimized', 'Features': n_selected, 'C-index': c_val},
])

print("\n" + comparison.to_string(index=False))

improvement_baseline = c_val - c_baseline
improvement_xgb = c_val - 0.7413

print(f"\n📊 Improvements:")
print(f"  vs V6.3 Baseline:  {improvement_baseline:+.4f}")
print(f"  vs V6.1 XGBoost:   {improvement_xgb:+.4f}")

if c_val > 0.7413:
    print(f"\n🎉 LightGBM optimized outperforms XGBoost!")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Performance comparison
models = comparison['Model'].tolist()
scores = comparison['C-index'].tolist()
colors = ['#808080', '#F18F01', '#3498DB', '#27AE60']

bars = ax1.bar(range(len(models)), scores, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_xticks(range(len(models)))
ax1.set_xticklabels(models, rotation=15, ha='right')

for bar, score in zip(bars, scores):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.4f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax1.axhline(y=0.74, color='orange', linestyle='--', linewidth=2, alpha=0.5)
ax1.set_ylabel('C-index', fontsize=12)
ax1.set_title('Model Performance', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Feature importance (top 15)
top_15 = importance_df.head(15)
ax2.barh(range(len(top_15)), top_15['importance'], color='steelblue')
ax2.set_yticks(range(len(top_15)))
ax2.set_yticklabels(top_15['feature'], fontsize=9)
ax2.invert_yaxis()
ax2.set_xlabel('Importance', fontsize=12)
ax2.set_title('Top 15 Features', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 9. Optimization History

In [ ]:
# Plot optimization history
fig, ax = plt.subplots(figsize=(12, 5))

trials_df = study.trials_dataframe()
ax.plot(trials_df['number'], trials_df['value'], alpha=0.6, linewidth=1, label='Trial C-index')
ax.plot(trials_df['number'], trials_df['value'].cummax(), 
        color='red', linewidth=2, label='Best C-index')

ax.set_xlabel('Trial', fontsize=12)
ax.set_ylabel('C-index', fontsize=12)
ax.set_title('Optuna Optimization History', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal best C-index: {study.best_value:.4f}")
print(f"Achieved at trial: {study.best_trial.number}")